# OpenAI Embeddings & Semantic Similarity Search

This interactive playground implements:
1. **OpenAI Embeddings** (`text-embedding-3-small`)
2. **Cosine Similarity** formula from scratch
3. **Pairwise Sentence Comparisons**
4. **Custom Semantic Search Engine** (`semantic_search` function)
5. Retrieval testing with natural language queries

In [ ]:
import os
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

# 1. Load environment variables
load_dotenv()

openai_api_key = os.getenv('OPENAI_API_KEY')
print(f"OPENAI_API_KEY detected: {'Yes' if openai_api_key and len(openai_api_key) > 10 else 'No (Please set OPENAI_API_KEY in .env)'}")

# 2. Initialize OpenAI Embeddings
try:
    embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
    test_vec = embeddings.embed_query('Test query')
    print(f"OpenAI Embeddings active! Vector dimension: {len(test_vec)}")
except Exception as exc:
    print(f"Notice: {exc}")
    print("Using remote HuggingFace embeddings as fallback for testing.")
    from langchain_huggingface import HuggingFaceEndpointEmbeddings
    embeddings = HuggingFaceEndpointEmbeddings(model='sentence-transformers/all-MiniLM-L6-v2')


## 1. Cosine Similarity Function
$$\text{similarity} = \frac{A \cdot B}{\|A\| \|B\|}$$

In [ ]:
def cosine_similarity(a, b) -> float:
    """Calculate cosine similarity between two vectors."""
    vec_a = np.array(a)
    vec_b = np.array(b)
    
    dot_product = np.dot(vec_a, vec_b)
    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)
    
    if norm_a == 0 or norm_b == 0:
        return 0.0
    
    return float(dot_product / (norm_a * norm_b))


## 2. Pairwise Sentence Comparisons

In [ ]:
sentences = [
    'The cat sat on the mat.',
    'A feline rested on the rug.',
    'The dog played in the yard.',
    'I love programming in Python.',
    'Python is my favorite programming language.'
]

sentence_embeddings = embeddings.embed_documents(sentences)
print(f"Generated embeddings for {len(sentences)} sentences.\n")

# Calculate similarity between all pairs (i, j)
for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        score = cosine_similarity(sentence_embeddings[i], sentence_embeddings[j])
        print(f"'{sentences[i]}' VS '{sentences[j]}'")
        print(f"  --> Similarity Score: {score:.4f}\n")


## 3. Semantic Search Engine from Scratch

In [ ]:
def semantic_search(query: str, documents: list[str], embedding_model, top_k: int = 3) -> list[tuple[float, str]]:
    """Retrieve top-k most semantically relevant documents for a query."""
    # 1. Embed query and document collection
    query_embedding = embedding_model.embed_query(query)
    doc_embeddings = embedding_model.embed_documents(documents)
    
    # 2. Compute similarity score for each document
    similarities = []
    for doc, doc_vec in zip(documents, doc_embeddings):
        score = cosine_similarity(query_embedding, doc_vec)
        similarities.append((score, doc))
    
    # 3. Sort by similarity descending
    similarities.sort(key=lambda x: x[0], reverse=True)
    
    # 4. Return top-k matches
    return similarities[:top_k]

# Document knowledge base
documents = [
    'LangChain is a framework for developing applications powered by language models.',
    'Python is a high-level programming language.',
    'Machine learning is a subset of artificial intelligence.',
    'Embeddings convert text into numerical vectors.',
    'The weather today is sunny and warm.'
]

print(f"Loaded {len(documents)} documents into knowledge base.")


## 4. Querying the Semantic Search Engine

In [ ]:
queries = [
    'What is LangChain?',
    'What is embeddings?'
]

for q in queries:
    print(f"\n==============================")
    print(f"Query: '{q}'")
    print(f"==============================")
    results = semantic_search(q, documents, embeddings, top_k=3)
    for rank, (score, doc) in enumerate(results, 1):
        print(f"Rank #{rank} (Similarity: {score:.4f}):")
        print(f"  {doc}")
